# Grove Vision AI V2 (Arm Ethos-U55) - Swift-YOLO Pipeline

**Target Device:** Seeed Studio Grove Vision AI Module V2 (Himax WiseEye2 HX6538)
**Compute Architecture:** Arm Cortex-M55 + Arm Ethos-U55 NPU (64 MACs/cycle)
**Framework Target:** TensorFlow Lite (INT8 Vela Compiled)


## ⚙️ Step 1: Environment Setup & Bulletproof Dependency Patching
This cell resolves the Colab Python 3.12 / PyTorch 2.11 NPU compilation errors by intercepting the `openmim` setuptools downgrade and forcing CUDA to bypass strict version checks. **Compilation will take 10-15 minutes.**

In [ ]:
import os

# 1. Clone Seeed Studio ModelAssistant (Stable 2.0.0 branch)
%cd /content
!rm -rf ModelAssistant
!git clone https://github.com/Seeed-Studio/ModelAssistant.git -b 2.0.0
%cd ModelAssistant

# 2. THE TROJAN HORSE FIX: Delete openmim from requirements to prevent setuptools destruction
!sed -i '/openmim/d' requirements/base.txt

# 3. Clean up broken packages and install stable build tools
!pip uninstall -y openmim openxlab setuptools
!pip install -q "setuptools==69.5.1" Cython numpy ninja wheel

# 4. Install OpenMMLab dependencies natively
!pip install -q ethos-u-vela "mmengine>=0.7.1" "mmdet==3.0.0"

# 5. Compile Modern MMCV (v2.2.0)
print("\n⏳ Compiling MMCV C++/CUDA ops... This will take 10-15 minutes. DO NOT INTERRUPT!")
os.environ['TORCH_ALLOW_CUDA_VERSION_MISMATCH'] = '1'
os.environ['MMCV_WITH_OPS'] = '1'
os.environ['MAX_JOBS'] = '4'
!pip install -q -v --no-build-isolation "mmcv>=2.2.0"

# 6. Safely patch MMDetection to accept MMCV 2.2.0+ (find the file WITHOUT importing mmdet, which would crash)
import importlib.util
spec = importlib.util.find_spec('mmdet')
mmdet_path = spec.submodule_search_locations[0]
init_file = os.path.join(mmdet_path, '__init__.py')
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '3.0.0'")
content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '3.0.0'")
with open(init_file, 'w') as f:
    f.write(content)
print("✅ MMDetection successfully patched!")

# 7. Install the rest of the ModelAssistant pipeline safely
!pip install -q -r requirements/base.txt
!pip install -q -r requirements/export.txt
!pip install -q -e .

# 8. Install the missing MMClassification library natively
!pip install -q "mmcls>=1.0.0rc6"
print("✅ MMClassification installed! You are clear for takeoff.")

# 9. Patch MMClassification to accept MMCV 2.2.0+
spec = importlib.util.find_spec('mmcls')
mmcls_path = spec.submodule_search_locations[0]
init_file = os.path.join(mmcls_path, '__init__.py')
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '3.0.0'")
content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '3.0.0'")
with open(init_file, 'w') as f:
    f.write(content)
print("✅ MMClassification successfully patched!")

# 10. Install the complete ONNX stack
!pip install -q onnx onnxruntime onnxsim onnxmltools onnxscript
print("✅ Complete ONNX stack installed!")

# 11. PyTorch 2.6+ Compatibility Patch (bypass weights_only default for train/export scripts)
patch_code = """
# --- PyTorch 2.6+ Compatibility Patch ---
import torch
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
# ----------------------------------------
"""

for script in ['tools/train.py', 'tools/export.py']:
    with open(script, 'r') as f:
        content = f.read()
    if "_safe_load" not in content:
        with open(script, 'w') as f:
            f.write(patch_code + '\n' + content)

print("✅ PyTorch security block bypassed! Training and Export scripts patched.")

print("\n✅ Environment is fully bulletproof and ready!")


## 📁 Step 2: Download Pretrained Weights & Custom Dataset

In [ ]:
%cd /content/ModelAssistant
%mkdir -p work_dirs/swift_yolo_192

# Download pretrained Swift-YOLO weights
!wget -q -c https://files.seeedstudio.com/sscma/model_zoo/detection/person/person_detection.pth -O work_dirs/swift_yolo_192/pretrain.pth

# Download and extract dataset (COCO format) with automatic overwrite (-o)
%mkdir -p work_dirs/swift_yolo_192/dataset
# REPLACE THE LINK BELOW WITH YOUR OWN ROBOFLOW COCO DATASET EXPORT LINK
!wget -q -c "YOUR_ROBOFLOW_EXPORT_LINK_HERE" -O work_dirs/swift_yolo_192/dataset.zip
!unzip -q -o work_dirs/swift_yolo_192/dataset.zip -d work_dirs/swift_yolo_192/dataset
print("\n✅ Dataset and pre-trained weights downloaded successfully!")

## 🚀 Step 3: Train / Fine-Tune Swift-YOLO

In [ ]:
%cd /content/ModelAssistant
!python3 tools/train.py configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
  --cfg-options \
    work_dir=work_dirs/swift_yolo_192 \
    num_classes=1 \
    epochs=100 \
    height=192 \
    width=192 \
    data_root=work_dirs/swift_yolo_192/dataset/ \
    load_from=work_dirs/swift_yolo_192/pretrain.pth

## 📦 Step 4: Export to Quantized TensorFlow Lite (INT8)

In [ ]:
import os
%cd /content/ModelAssistant

# Resolve last trained checkpoint path
checkpoint_file = 'work_dirs/swift_yolo_192/last_checkpoint'
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint_path = f.read().strip()
else:
    checkpoint_path = 'work_dirs/swift_yolo_192/epoch_100.pth'

print(f"Using checkpoint: {checkpoint_path}")

# Export to PyTorch, ONNX, and INT8 TFLite
!python3 tools/export.py configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
  "{checkpoint_path}" \
  --cfg-options \
    work_dir=work_dirs/swift_yolo_192 \
    num_classes=1 \
    height=192 \
    width=192 \
    data_root=work_dirs/swift_yolo_192/dataset/

## ⚡ Step 5: Arm Ethos-U55 Hardware NPU Compilation (Vela)

In [ ]:
import glob
%cd /content/ModelAssistant

# Find the raw INT8 TFLite model and compile it specifically for Grove Vision AI V2
tflite_models = glob.glob("work_dirs/swift_yolo_192/*_int8.tflite")
if tflite_models:
    raw_int8_model = tflite_models[0]
    print(f"Compiling {raw_int8_model} with Vela...")
    !vela {raw_int8_model} \
      --accelerator-config ethos-u55-64 \
      --optimise Size \
      --memory-mode Shared_Sram \
      --system-config Ethos_U55_High_End_Embedded \
      --output-dir work_dirs/swift_yolo_192/
else:
    print("No INT8 TFLite model found for Vela compilation!")

## 📊 Step 6: Verification

In [ ]:
%cd /content/ModelAssistant
print("\n--- Generated Compiled Artifacts ---")
!ls -lh work_dirs/swift_yolo_192/*_vela.tflite